# 实验任务一: Transformer

## **1. Transformer 编码器中 Encoder Layer 的实现**
Encoder Layer简介
    Encoder Layer 是 Transformer 编码器中的基本构建单元，由 多头自注意力机制（Multi-Head Self-Attention） 和 前馈全连接网络（Feed Forward Network） 组成，搭配两次残差连接与 LayerNorm，用于高效建模输入序列的上下文依赖关系和特征表达能力。


这次我们还是使用AG News 数据集进行后续的分类任务，由于读取数据方式的改变，需要重新下载一下数据集。


AG News 数据集简介

    AG News 数据集来源于 AG's corpus of news articles，是一个大型的新闻数据集，由 Antonio Gulli 从多个新闻网站收集整理。
    AG News 数据集包含 4 类新闻，每类 30,000 条训练数据，共 120,000 条训练样本 和 7,600 条测试样本。

首先导入所需模块：




In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
# from torchtext.data.utils import get_tokenizer
# from torchtext.vocab import build_vocab_from_iterator
from tqdm import tqdm
import pandas as pd
from transformers import AutoTokenizer

/root/miniconda3/envs/pytorch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


数据读取以及预处理

不同于上次，这次我们使用pandas读取数据，相应的代码也有所修改。

In [3]:
# **1. 加载 AG NEWS 数据集**
df = pd.read_csv("/root/DL-lab/lab5/ag/train.csv")  # 请替换成你的文件路径
df.columns = ["label", "title", "description"]  # CSV 有3列: 标签, 标题, 描述
df["text"] = df["title"] + " " + df["description"]  # 合并标题和描述作为输入文本
df["label"] = df["label"] - 1  # AG NEWS 的标签是 1-4，我们转换成 0-3
train_texts, train_labels = df["text"].tolist(), df["label"].tolist()
number = int(0.3 * len(train_texts))
train_texts, train_labels = train_texts[: number], train_labels[: number]

df = pd.read_csv("/root/DL-lab/lab5/ag/test.csv")  # 请替换成你的文件路径
df.columns = ["label", "title", "description"]  # CSV 有3列: 标签, 标题, 描述
df["text"] = df["title"] + " " + df["description"]  # 合并标题和描述作为输入文本
df["label"] = df["label"] - 1  # AG NEWS 的标签是 1-4，我们转换成 0-3
test_texts, test_labels = df["text"].tolist(), df["label"].tolist()

# **2. 加载 BERT Tokenizer**
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# **3. 处理数据**
class AGNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=50):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text, truncation=True, padding="max_length", max_length=self.max_length, return_tensors="pt"
        )
        input_ids = encoding["input_ids"].squeeze(0)
        return input_ids, torch.tensor(label, dtype=torch.long)

vocab = tokenizer.get_vocab()
pad_idx = tokenizer.pad_token_id
unk_idx = tokenizer.unk_token_id

train_dataset = AGNewsDataset(train_texts, train_labels, tokenizer)
test_dataset = AGNewsDataset(test_texts, test_labels, tokenizer)

train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)


### 位置编码器
在 Transformer 模型中，Self-Attention 是无序的，它无法感知输入序列的「位置信息」，即每个 token 在序列中的先后顺序。

相比之下，RNN（通过时间步）和 CNN（通过局部感受野）天然就有顺序/位置的概念。

因此，Transformer 需要为每个 token embedding 加入「位置信息」，这就是 Positional Encoding 的作用。

论文《Attention is All You Need》中提出了如下的位置编码方式，使用正弦和余弦函数来构造具有不同频率的位置表示。

原始公式：

$$
PE(pos, 2i) = \sin\left( \frac{pos}{10000^{\frac{2i}{d_{model}}}} \right)
$$

$$
PE(pos, 2i+1) = \cos\left( \frac{pos}{10000^{\frac{2i}{d_{model}}}} \right)
$$

其中：

-  $pos$：是序列中的位置$（0, 1, 2, ..., L-1）$
-  $i$：是 embedding 的维度索引$（0 ~ d_{model}/2）$
-  $d_{model}$：是 embedding 的总维度
-  $10000$：是一个控制不同维度的 $sin/cos$ 波动频率的超参数


公式进一步推导，
原公式中：

$$
\frac{pos}{10000^{\frac{2i}{d_{model}}}}
$$

等价于：

$$
pos \times \frac{1}{10000^{\frac{2i}{d_{model}}}}
$$

进一步展开成指数形式：

$$
= pos \times e^{- \log(10000) \cdot \frac{2i}{d_{model}}}
$$

请参考展开后的指数形式补充完下面的代码：

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(torch.log(torch.tensor(10000.0)) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer('pe', pe.unsqueeze(0)) 

    def forward(self, x):
        x = x + self.pe[:, :x.size(1)]
        return x


    思考题1：为什么需要对偶数和奇数维度分别使用 sin 和 cos？

### Multi-Head Self-Attention 模块

下面是多头自注意力的实现，请你按照要求补全代码：

In [6]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_k = d_model // n_heads
        self.n_heads = n_heads

        self.qkv_linear = nn.Linear(d_model, d_model * 3)

        self.fc = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        batch_size, seq_len, d_model = x.size()

        qkv = self.qkv_linear(x)

        qkv = qkv.view(batch_size, seq_len, self.n_heads, 3 * self.d_k)

        qkv = qkv.permute(0, 2, 1, 3)

        q, k, v = qkv.chunk(3, dim=-1) 

        scores = torch.matmul(q, k.transpose(-2, -1)) / (self.d_k ** 0.5)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        attn = torch.softmax(scores, dim=-1)

        context = torch.matmul(attn, v)

        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, d_model)

        return self.fc(context)


    思考题2：在 Multi-Head Self-Attention 机制中，为什么我们需要使用多个 attention head？

    思考题3：为什么要用缩放因子 sqrt(d_k)？

### TransformerEncoderLayer

下面的代码实现了 Transformer 编码器中的一个标准 Encoder Layer，包含：

“多头自注意力 + 前馈网络 + 两次残差连接 + 两次 LayerNorm” 的结构，用于对输入序列进行特征建模和上下文信息融合。

请你按照要求补全代码：

In [7]:
class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()

        self.self_attn = MultiHeadSelfAttention(d_model, n_heads)

        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Linear(d_ff, d_model)
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x, mask=None):
        x2 = self.self_attn(x, mask)
        x = self.norm1(x + x2)
        
        x2 = self.ff(x)
        x = self.norm2(x + x2)

        return x


    思考题4：为什么 Transformer Encoder Layer 中要在 Self-Attention 和 Feed Forward Network 之后都使用残差连接和 LayerNorm？试从“模型训练稳定性”和“特征传递”两个角度进行分析。

## **2. 基于 Transformer Encoder 的文本分类器**

下面，我们实现一个基于 Transformer Encoder 的文本分类器，通过 embedding、位置编码、多层 encoder 处理输入序列，最终使用 mean pooling 和全连接层完成文本的多类别分类任务。


In [8]:
class TransformerEncoderClassifier(nn.Module):
    def __init__(self, vocab_size, d_model=128, n_heads=4, d_ff=256, num_layers=2, num_classes=4):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_idx)
        self.pos_encoder = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([TransformerEncoderLayer(d_model, n_heads, d_ff) for _ in range(num_layers)])
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoder(x)
        pad_mask = (x.sum(-1) != 0).unsqueeze(1).unsqueeze(2)

        for layer in self.layers:
            x = layer(x, pad_mask)
            
        out = x.mean(dim=1)

        return self.fc(out)


    思考题5：为什么在 TransformerEncoderClassifier 中，通常会在 Encoder 的输出上做 mean pooling（对 seq_len 取平均）？除了 mean pooling，你能否想到其他可以替代的 pooling 或特征聚合方式？并简要分析它们的优缺点。

下面是模型的训练和测试:

In [9]:
# 使用 split 进行分词
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransformerEncoderClassifier(len(vocab)).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

模型训练部分：

In [10]:
def train_epoch():
    model.train()
    total_loss = 0
    loop = tqdm(train_dataloader, desc="Training", leave=False)
    for text, labels in loop:
        text, labels = text.to(device), labels.to(device)
        optimizer.zero_grad()
        output = model(text)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        # 更新tqdm进度条
        loop.set_postfix(loss=loss.item())
    return total_loss / len(train_dataloader)

模型测试部分：

In [11]:
def evaluate():
    model.eval()
    correct = 0
    total = 0
    loop = tqdm(test_dataloader, desc="Evaluating", leave=False)
    with torch.no_grad():
        for text, labels in loop:
            text, labels = text.to(device), labels.to(device)
            output = model(text)
            preds = output.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total


for epoch in range(1, 6):
    loss = train_epoch()
    acc = evaluate()
    print(f'Epoch {epoch}: Loss = {loss:.4f}, Test Acc = {acc:.4f}')

Epoch 1: Loss = 0.5841, Test Acc = 0.8282


Epoch 2: Loss = 0.3515, Test Acc = 0.8480


Epoch 3: Loss = 0.2840, Test Acc = 0.8593


Epoch 4: Loss = 0.2374, Test Acc = 0.8558


Epoch 5: Loss = 0.2051, Test Acc = 0.8617



    思考题6：Transformer 相比传统的 RNN/CNN，优势在哪里？为什么 Transformer 更适合处理长文本？